In [9]:
import numpy as np
import rioxarray 
import rasterio as rio
from rasterio.plot import show
from rasterio.enums import Resampling
import glob
import os
from osgeo import gdal
from pathlib import Path

from win32comext.axscript.client.framework import profile



In [56]:
LiDAR_dir = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR"
raster_files = glob.glob(os.path.join(LiDAR_dir, '*.tif'))

In [46]:
##Using GDAL
extent = (601558.000, 4862467.500, 609431.500, 4870872.500)

for raster in raster_files:
    out_dir = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/resampled"
    name, ext = os.path.splitext(os.path.basename(raster))
    out_raster = os.path.join(out_dir, f"{name}_100{ext}")
    gdal.Warp(
        out_raster, 
        raster, 
        dstNodata=float("nan"),
        xRes=100,
        yRes=100,
        resampleAlg="average",
        outputBounds=extent)

C:\Users\RDCRLSMC\AppData\Local\miniconda3\envs\iceroad\lib\site-packages\osgeo\gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [52]:
src_agg = glob.glob(os.path.join(LiDAR_dir, '*100*'))

for raster in src_agg:
    print(raster)

C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20221208_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20230209_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20230316_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20230405_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20231228_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20240115_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20240213_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20240315_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20240418_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20250113_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20250129_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20250404_SD_V01.0_100.tif
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20250501_SD_

In [55]:
for raster in src_agg:
    with rio.open(raster) as src:
        data = src.read(1, masked=True)  # masked array respects nodata

        # Flatten valid values
        vals = data.compressed()

        low_thresh = np.percentile(vals, 1)
        high_thresh = np.percentile(vals, 99)
        
        print(raster)
        print("Lowest 1% threshold:", low_thresh)
        print("Highest 1% threshold:", high_thresh)


C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20221208_SD_V01.0_100.tif
Lowest 1% threshold: 0.38710456162691115
Highest 1% threshold: 1.37091171503067
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20230209_SD_V01.0_100.tif
Lowest 1% threshold: 0.6983704298734665
Highest 1% threshold: 2.3634479427337647
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20230316_SD_V01.0_100.tif
Lowest 1% threshold: 1.0224434697628022
Highest 1% threshold: 3.482961461544037
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20230405_SD_V01.0_100.tif
Lowest 1% threshold: 1.2952988398075105
Highest 1% threshold: 4.031616425514216
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20231228_SD_V01.0_100.tif
Lowest 1% threshold: 0.026102729700505724
Highest 1% threshold: 1.1208970570564272
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Lidar_20240115_SD_V01.0_100.tif
Lowest 1% threshold: 0.471787366271019
Highest 1% threshold: 1.5706566691398622
C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR\SNEX_MCS_Li

In [19]:
#Using Rasterio
for raster in raster_files:
    dir_name = os.path.dirname(raster)
    name, ext = os.path.splitext(os.path.basename(raster))
    out_raster = os.path.join(dir_name, f"{name}_100_rasterio{ext}")
    sd = rioxarray.open_rasterio(raster)
    sd.rio.write_nodata(np.nan, inplace=True)  # make sure nans are counted as no data
    sd_downsampled = sd.rio.reproject(
        sd.rio.crs,
        resolution = (100,100),
        resampling=Resampling.average)
    sd_downsampled.rio.to_raster(out_raster)





## Decided to go back to LiDAR point clouds
### Resampled the following output: /ice-road/results/pc-transform/laz-snow-trans_source.laz
used following pipeline on Borah:\
        [ \
    { \
        "type": "readers.las", \
        "filename": "$laz"\
    }, \
    { \
        "type": "writers.gdal", \
        "filename": "$tif", \
        "resolution": 100, \
        "output_type": "mean", \
        "dimension": "Z", \
        "data_type": "float32", \
        "nodata": -9999 \
    } \
] \
Outputs saved to C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR2

In [12]:

dir = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR2")
tifs = glob.glob(os.path.join(dir, "*.tif"))
ref = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR2/ref/MCS_REFDEM_WGS84_cubic.tif"

In [24]:
with rio.open(ref) as ref_src:
    ref_dem = ref_src.read(1, masked=True)
    ref_profile = ref_src.profile
    ref_width = ref_src.width
    ref_height = ref_src.height
    ref_transform = ref_src.transform
    print(ref_profile)

{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -3.4028234663852886e+38, 'width': 79, 'height': 84, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 601557.9999991942,
       0.0, -100.0, 4870872.500118704), 'blockysize': 25, 'tiled': False, 'interleave': 'band'}


In [21]:
for raster in tifs:
    with rio.open(raster) as src:
        src.read(1, masked=True)
        print(src.profile)


{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -9999.0, 'width': 80, 'height': 85, 'count': 1, 'crs': CRS.from_wkt('COMPD_CS["WGS 84 / UTM zone 11N",PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32611"]],VERT_CS["Ellipsoidal Heights",VERT_DATUM["unknown",2005],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Up",UP]]]'), 'transform': Affine(100.0, 0.0, 601474.81,
       0.0, -100.0, 4870883.3100000005), 'blockysize': 25, 'tiled': False, 'interleave':

In [41]:
for raster_file in tifs:
    with rio.open(raster_file) as src:
        # Resample to match reference dimensions
        data = src.read(1,
            out_shape=(ref_height, ref_width),
            resampling=Resampling.bilinear  # DEMs usually bilinear
        )

        # Update profile for new raster
        out_profile = ref_profile
        out_profile.update({
            "dtype": data.dtype,
            "height": ref_height,
            "width": ref_width,
            "transform": ref_transform,
            "nodata": -9999
        })

        # Write aligned raster
        resampled_dir = os.path.join(dir, "resampled")
        os.makedirs(resampled_dir, exist_ok=True)  # creates it if it doesn't exist
        out_path = os.path.join(resampled_dir, os.path.basename(raster_file))
        with rio.open(out_path, "w", **out_profile) as dst:
            dst.write(data, 1)

print("All rasters aligned to reference raster.")

All rasters aligned to reference raster.


In [42]:
for raster in glob.glob(os.path.join('C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR2/resampled', "*.tif")):
    with rio.open(raster) as src:
        src.read(1, masked=True)
        print(src.profile)

{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -9999.0, 'width': 79, 'height': 84, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 601557.9999991942,
       0.0, -100.0, 4870872.500118704), 'blockysize': 25, 'tiled': False, 'interleave': 'band'}
{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -9999.0, 'width': 79, 'height': 84, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 601557.9999991942,
       0.0, -100.0, 4870872.500118704), 'blockysize': 25, 'tiled': False, 'interleave': 'band'}
{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -9999.0, 'width': 79, 'height': 84, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 601557.9999991942,
       0.0, -100.0, 4870872.500118704), 'blockysize': 25, 'tiled': False, 'interleave': 'band'}
{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -9999.0, 'width': 79, 'height': 84, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 601557.9999991942,


In [44]:
for raster in glob.glob(os.path.join(resampled_dir, "*.tif")): 
    with rio.open(raster) as src:
        base = os.path.basename(raster).split("-")[0]   
        out_raster = os.path.join(dir, f"{base}_sd.tif")
        
        # Read DTM and metadata
        dtm = src.read(1, masked=True)
        dsm_meta = src.profile

        # Compute snow depth
        sd = dtm - ref_dem

        # Write output raster
        with rio.open(out_raster, "w", **dsm_meta) as dst:
            dst.write(sd.astype("float32"), 1)

### Some of these rasters deviate from the reference raster by over 100 (close to 200) m. Therefore, returning to manipulation of 0.5-m rasters
### I think there is an issue with averaging the point cloud over a 100-m region when there are data gaps 

In [97]:
LiDAR_dir = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR")
raster_files = glob.glob(os.path.join(LiDAR_dir, '*.tif'))

for raster in raster_files:
    with rio.open(raster) as src:
        sd = src.read(1, masked=True)
        #print(sd.max(), sd.min())
        print(src.shape)

(16810, 15747)
(16810, 15747)
(16810, 15747)
(16810, 15747)

KeyboardInterrupt: 

In [98]:
#Remove below 0 values and remove highest 1% of values (outliers)

upscale_factor = 200

for raster in raster_files:
    with rio.open(raster) as src:
        out_raster = os.path.join(LiDAR_dir / "filtered", os.path.basename(raster).replace(".tif", "_filtered.tif"))
        #print(out_raster)
        print(src.shape)
        sd = src.read(1, masked=True).filled(np.nan)  # <-- fill masked with NaN
        sd[sd < 0] = np.nan
        vals = sd[np.isfinite(sd)]
        upper = np.percentile(vals, 99.9)
        sd[sd > upper] = np.nan
        #print(np.nanmax(sd), np.nanmin(sd))
        meta = src.profile
        # Write output raster
        with rio.open(out_raster, "w", **meta) as dst:
            dst.write(sd.astype("float32"), 1)


#Resample rasters to 100-m

for raster in glob.glob(os.path.join(LiDAR_dir / "filtered", "*.tif")):
    dir_name = os.path.dirname(raster)
    name, ext = os.path.splitext(os.path.basename(raster))
    out_raster = os.path.join(dir_name, f"{name}_100_rasterio{ext}")
    sd = rioxarray.open_rasterio(raster)
    sd.rio.write_nodata(np.nan, inplace=True)  # make sure nans are counted as no data
    sd_downsampled = sd.rio.reproject(
        sd.rio.crs,
        resolution = (100,100),
        resampling=Resampling.average)
    sd_downsampled.rio.to_raster(out_raster)        

(16810, 15747)
1.9680176 0.0
(16810, 15747)
3.1086426 0.0
(16810, 15747)
4.5625 0.0
(16810, 15747)
7.946289 0.0
(16810, 15747)
1.7050781 0.0
(16810, 15747)
2.4490967 0.0
(16810, 15747)
2.8293457 0.0
(16810, 15747)
3.6252441 0.0
(16810, 15747)
3.4035645 0.0
(16810, 15747)
3.1486816 0.0
(16810, 15747)
2.744629 0.0
(16810, 15747)
4.529541 0.0
(16810, 15747)
3.8205566 0.0


In [ ]:
## Try moving window approach

import sys
import numpy as np
from scipy import ndimage
from osgeo import gdal
from osgeo.gdalconst import *

#Setup variables
INPUT_DATA= "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/test/SNEX_MCS_Lidar_20250404_SD_V01.0.tif"
OFILE= "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/test/SNEX_MCS_Lidar_20250404_SD_V01.0_mw.tif"
KERNEL_DIMENSIONS=[100,100] # doesn't have to be square

#Read in with gdal
inDs = gdal.Open(INPUT_DATA)

#Get data and geotransform info
geotransform = inDs.GetGeoTransform()
rows = inDs.RasterYSize
cols = inDs.RasterXSize
array2process = np.array(inDs.ReadAsArray())

#Sort array in case of NAN values
workarr = np.nan_to_num(array2process)

#Set up function 
def focal_variance(subarr):
	"""Calcualte variance of an array
	"""
	return(ndimage.variance(subarr))

#Set up moving window (kernel) dimensions
footprint=np.ones((KERNEL_DIMENSIONS[0],KERNEL_DIMENSIONS[1])) 

#Apply function across dataset
processedArray=ndimage.generic_filter(workarr, focal_variance, footprint=footprint)

#~~~~~~~~~~~~~~~~~
#Write out dataset, assigning geotransform info of the input dataset
# - assumes that output array has the same dimensions as the input array

#Create the output gdal object
driver = inDs.GetDriver()

outDs = driver.Create(OFILE, cols, rows, 1, GDT_Float32)
if outDs is None:
    sys.exit("Unable to create %s" %OFILE)

outBand = outDs.GetRasterBand(1)

# write the data
outBand.WriteArray(processedArray)

# georeference the data and set the projection based on input dataset
outDs.SetGeoTransform(inDs.GetGeoTransform())
outDs.SetProjection(inDs.GetProjection())
outBand.SetNoDataValue(np.nan)

outBand.FlushCache()